# BuffData: Accuracy-Gated Generation Benchmark

This notebook runs BuffData's `AccuracyGatedGenerator` on a real Hugging Face dataset and
verifies the claim end to end: **a classifier trained on the generated dataset must score
at least X% higher (relative), on a held-out validation set, than the same classifier
trained on the original data** -- or the run reports an honest shortfall instead of a
fabricated pass.

**What you need before running this:**
1. `buffdata_colab.zip` (source-only export of the repo, no data files) -- uploaded in the next cell.
2. An API key for one LLM provider (Gemini, OpenAI, or Anthropic) -- entered further down, never printed or saved to disk.

Runtime: CPU is enough. The accuracy gate trains a tiny `EmbeddingBag + Linear` proxy
classifier (not the LLM) to measure accuracy; that step takes seconds per round regardless
of whether you're on CPU or GPU.

## 1. Upload the BuffData source

Run this cell, then pick `buffdata_colab.zip` when prompted. If you don't have it yet,
create it from your machine with:
```bash
cd ~/projects/buffdata
zip -r buffdata_colab.zip buffdata pyproject.toml README.md examples \
  benchmarks/benchmark_buffdata.py benchmarks/benchmark_accuracy_regression.py \
  benchmarks/benchmark_strict_scale.py benchmarks/README.md \
  -x '*/__pycache__/*' -x '*.pyc'
```

In [ ]:
from google.colab import files
import os, zipfile

if not os.path.exists('/content/buffdata/pyproject.toml'):
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall('/content')
    print('Extracted to /content/buffdata')
else:
    print('BuffData source already present, skipping upload.')

In [ ]:
%cd /content/buffdata
!pip install -q -e ".[dev]"
print('Install complete.')

## 2. Configure the LLM provider

Enter the API key for exactly one provider. `getpass` keeps it out of the notebook's
saved output and off disk; it only lives in this process's environment variables.

In [ ]:
import getpass, os

PROVIDER = 'gemini'  # 'gemini', 'openai', or 'anthropic'

key_env = {'gemini': 'GEMINI_API_KEY', 'openai': 'OPENAI_API_KEY', 'anthropic': 'ANTHROPIC_API_KEY'}[PROVIDER]
os.environ[key_env] = getpass.getpass(f'Enter your {PROVIDER} API key: ')
print(f'{key_env} set for this session.')

## 3. Build the baseline ("old") dataset and a held-out validation set

Uses AG News from Hugging Face. `TRAIN_ROWS` is deliberately small so the baseline
classifier is under-trained -- that's the realistic case BuffData's generation step is
meant to help. `VALIDATION_ROWS` is a separate, disjoint stratified sample used only for
scoring, never for training or generation.

In [ ]:
import random
from datasets import load_dataset
from buffdata.models.schemas import DatasetItem

TRAIN_ROWS = 300
VALIDATION_ROWS = 400
CLASSES = 4  # AG News: World, Sports, Business, Sci/Tech

def stratified_rows(split, count, classes, seed):
    buckets = {label: [] for label in range(classes)}
    for row in split:
        buckets[int(row['label'])].append({'text': str(row['text']).strip(), 'label': int(row['label'])})
    rng = random.Random(seed)
    per_class, remainder = divmod(count, classes)
    result = []
    for label in range(classes):
        bucket = buckets[label]
        rng.shuffle(bucket)
        take = per_class + (1 if label < remainder else 0)
        result.extend(bucket[:take])
    rng.shuffle(result)
    return result

print('Downloading AG News...')
source = load_dataset('fancyzhx/ag_news')
train_rows = stratified_rows(source['train'], TRAIN_ROWS, CLASSES, seed=7)
validation_rows = stratified_rows(source['test'], VALIDATION_ROWS, CLASSES, seed=11)

original_items = [DatasetItem.from_dict(row) for row in train_rows]
validation_items = [DatasetItem.from_dict(row) for row in validation_rows]
print(f'{len(original_items)} training rows, {len(validation_items)} validation rows.')

## 4. Run the accuracy-gated generator

Target: at least 10% relative accuracy gain over the baseline, within 5 rounds. Each
round that misses the gate feeds the proxy classifier's weakest classes and its actual
misclassified validation examples back into the next round's generation prompt.

In [ ]:
from buffdata.engine.client import create_llm_client
from buffdata.engine.limiter import AsyncRateLimiter
from buffdata.optimizers.gated_generator import AccuracyGatedGenerator

client = create_llm_client(PROVIDER)
limiter = AsyncRateLimiter(max_rpm=30, concurrency=4)
generator = AccuracyGatedGenerator(client=client, limiter=limiter)

generated_items, report = await generator.generate(
    original_items,
    validation_items,
    min_relative_gain=0.10,
    max_iterations=5,
    multiplier=2,
    chunk_size=20,
    accuracy_seeds=[17, 29, 43],
    accuracy_epochs=6,
)

print(f"Gate {'PASSED' if report.passed else 'FAILED'} after {report.iterations_used} round(s)")
print(f"Baseline accuracy: {report.baseline_accuracy:.4f}")
print(f"Final accuracy:    {report.final_accuracy:.4f}")
print(f"Relative gain:     {report.relative_gain:+.1%} (target {report.target_relative_gain:.0%})")
print(f"Generated pool:    {len(generated_items)} rows (from {len(original_items)} originals)")

## 5. Per-round detail and chart

In [ ]:
import pandas as pd

rows = [
    {
        'round': metric.iteration,
        'pool_size': metric.pool_size,
        'accuracy': metric.accuracy,
        'relative_gain': metric.relative_gain,
        'weak_classes': ', '.join(metric.weak_labels) or '-',
        'gate': 'PASS' if metric.accepted else 'FAIL',
    }
    for metric in report.per_iteration
]
df = pd.DataFrame(rows)
df

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(
    ['Original', 'Generated'],
    [report.baseline_accuracy, report.final_accuracy],
    color=['#9e9e9e', '#2e7d32' if report.passed else '#c62828'],
)
ax.set_ylabel('Held-out accuracy')
ax.set_ylim(0, 1)
ax.set_title(f"{report.relative_gain:+.1%} relative gain vs. {report.target_relative_gain:.0%} target")
for bar, value in zip(bars, [report.baseline_accuracy, report.final_accuracy]):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.01, f'{value:.3f}', ha='center')
plt.show()

## 6. Save and download the generated dataset

In [ ]:
from buffdata.models.formats import write_dataset
import json

write_dataset(generated_items, '/content/generated.jsonl')
with open('/content/accuracy_gate_report.json', 'w') as handle:
    handle.write(report.model_dump_json(indent=2))

files.download('/content/generated.jsonl')
files.download('/content/accuracy_gate_report.json')

## 7. Optional: full multi-dataset README-style benchmark

This runs the repo's existing (heavier) `benchmarks/benchmark_buffdata.py`, which trains
on real clean and deliberately-corrupted samples from several Hugging Face datasets and
reproduces the accuracy-recovery table style used in `README.md`. It's slower and not
required for the accuracy-gated-generation check above.

In [ ]:
# !python benchmarks/benchmark_buffdata.py --output-dir benchmarks/results --train-rows 3000 --test-rows 1000 \
#   --datasets ag_news --gemini-audit-rows 20